In [2]:
import os
import fiona
import geopandas as gpd
import pandas as pd
import numpy as np
import pyarrow as pa

In [21]:
solar_gml_path = "./maps/Solarpotenzial/wfs_solarpotenzialanalyse"
GOOD_EIGNUNG = {
    "Eignung 1",
    "Eignung 2",
    "Eignung 3",
}

In [22]:
gdf_dach = gpd.read_file(solar_gml_path, layer="dachseiten")
print(gdf_dach.head())

                        gml_id    area  aspect  aufstd  buildingid  \
0  DE.HH.UP_DACHSEITEN_1745005    8.42  132.63       1        3398   
1  DE.HH.UP_DACHSEITEN_1745006   24.33  170.43       1        3568   
2  DE.HH.UP_DACHSEITEN_1745007   50.22   48.11       1        3609   
3  DE.HH.UP_DACHSEITEN_1745008   63.65  206.10       1        3613   
4  DE.HH.UP_DACHSEITEN_1745009  562.53  276.67       1        3653   

                                             eignung  \
0  Eignung 6 (geringe Einstrahlung / weniger geei...   
1  Eignung 6 (geringe Einstrahlung / weniger geei...   
2        Eignung 3 (geeignet, mittlere Einstrahlung)   
3            Eignung 2 (geeignet, hohe Einstrahlung)   
4  Eignung 6 (geringe Einstrahlung / weniger geei...   

                                           eignung_t  ertkwp_k  ertkwp_ka  \
0  Eignung 6 (geringe Einstrahlung / weniger geei...      0.00       0.00   
1  Eignung 6 (geringe Einstrahlung / weniger geei...      0.00       0.00   
2        Ei

In [23]:
gdf_geb = gpd.read_file(solar_gml_path, layer="gebaeude")
print(gdf_geb.head())

                     gml_id                               address  \
0  DE.HH.UP_GEBAEUDE_769817                                         
1  DE.HH.UP_GEBAEUDE_769818             Neuer Steinweg 2, Hamburg   
2  DE.HH.UP_GEBAEUDE_769819                                         
3  DE.HH.UP_GEBAEUDE_769820  Simon-von-Utrecht-Straße 66, Hamburg   
4  DE.HH.UP_GEBAEUDE_769821                                         

                                        eignung  \
0       Eignung 2 (geeignet, hohe Einstrahlung)   
1       Eignung 2 (geeignet, hohe Einstrahlung)   
2  Eignung 1 (geeignet, sehr hohe Einstrahlung)   
3  Eignung 1 (geeignet, sehr hohe Einstrahlung)   
4  Eignung 1 (geeignet, sehr hohe Einstrahlung)   

                                      eignung_t  \
0       Eignung 2 (geeignet, hohe Einstrahlung)   
1       Eignung 2 (geeignet, hohe Einstrahlung)   
2  Eignung 1 (geeignet, sehr hohe Einstrahlung)   
3  Eignung 1 (geeignet, sehr hohe Einstrahlung)   
4  Eignung 1 (geeignet,

In [31]:
solar_df = gdf_dach.copy()

solar_df = gpd.GeoDataFrame(solar_df, geometry="geometry", crs="EPSG:25832")
solar_df["is_suitable"] = solar_df["eignung"].str.contains("Eignung 1|Eignung 2|Eignung 3", na=False)

# calculate usable area
solar_df["usable_roof_area"] = (solar_df["area"] * solar_df["is_suitable"].astype(float))

# fix invalid geometry
solar_df["geometry"] = solar_df["geometry"].buffer(0)

# join the geometry of usable roof parts
buildings_geom = (
    solar_df[solar_df["is_suitable"]]
    .groupby("buildingid")["geometry"]
    .apply(lambda x: x.union_all())
    .reset_index()
)

# join the usable area data
buildings_stats = (
    solar_df
    .groupby("buildingid", as_index=False)
    .agg(
        usable_roof_area=("usable_roof_area", "sum"),
        area=("area", "sum")
    )
)

# merge stats and geometrie
buildings_solar = buildings_geom.merge(buildings_stats, on="buildingid", how="left")

# fill no data
buildings_solar["usable_roof_area"] = buildings_solar["usable_roof_area"].fillna(0)
buildings_solar = buildings_solar.set_crs("EPSG:25832")
buildings_solar.to_parquet("buildings_solar_with_geometry.parquet")

print(buildings_solar.head())
print(buildings_solar.crs)

   buildingid                                           geometry  \
0          14  POLYGON ((562361.274 5927756.7, 562361.395 592...   
1          16  POLYGON ((559696.435 5932404.392, 559696.632 5...   
2          22  POLYGON ((558679.379 5938246.673, 558678.202 5...   
3          26  POLYGON ((573898.217 5926177.412, 573897.157 5...   
4          59  MULTIPOLYGON (((561072.151 5931902.08, 561072....   

   usable_roof_area    area  
0              6.10    6.10  
1             16.01   16.01  
2             30.79   43.22  
3             43.03   43.03  
4            527.14  557.70  


In [33]:
gml_dir = "./maps/ALKIS_Liegenschaftskarte_ausgewaehlteDaten_HH_2026-01-15/" 

# All available layers in ALKIS GML files (from a single example so there is more in other files probs.)
ALKIS_LAYERS = [
    'AP_LPO', 'AP_PPO', 'AP_PTO',
    'AX_Baublock', 'AX_BauRaumOderBodenordnungsrecht', 'AX_Bauteil',
    'AX_BauwerkImGewaesserbereich', 'AX_BauwerkImVerkehrsbereich',
    'AX_BauwerkOderAnlageFuerSportFreizeitUndErholung',
    'AX_BesondereFlurstuecksgrenze', 'AX_BesondereGebaeudelinie',
    'AX_Bodenschaetzung', 'AX_DammWallDeich', 'AX_Denkmalschutzrecht',
    'AX_FlaecheBesondererFunktionalerPraegung', 'AX_FlaecheGemischterNutzung',
    'AX_Fliessgewaesser', 'AX_Flurstueck', 'AX_Gebaeude', 'AP_Darstellung',
    'AX_Gehoelz', 'AX_GeoreferenzierteGebaeudeadresse',
    'AX_GrablochDerBodenschaetzung', 'AX_IndustrieUndGewerbeflaeche',
    'AX_KlassifizierungNachWasserrecht', 'AX_Landwirtschaft', 'AX_Platz',
    'AX_Schiffsverkehr', 'AX_SonstigesBauwerkOderSonstigeEinrichtung',
    'AX_SonstigesRecht', 'AX_SportFreizeitUndErholungsflaeche',
    'AX_StehendesGewaesser', 'AX_Strassenverkehr', 'AX_Strassenverkehrsanlage',
    'AX_Strukturlinie3D', 'AX_UnlandVegetationsloseFlaeche',
    'AX_UntergeordnetesGewaesser', 'AX_Vegetationsmerkmal',
    'AX_VorratsbehaelterSpeicherbauwerk', 'AX_Wald', 'AX_Weg',
    'AX_WegPfadSteig', 'AX_Wohnbauflaeche'
]

# Columns available in AX_Gebaeude (from a single example so they are diff. in other files probs.)
AX_GEBAEUDE_COLUMNS = [
    'identifier', 'beginnt', 'advStandardModell', 'gebaeudefunktion',
    'weitereGebaeudefunktion', 'name', 'bauweise', 'anzahlDerOberirdischenGeschosse',
    'anzahlDerUnterirdischenGeschosse', 'hochhaus', 'objekthoehe',
    'dachform', 'zustand', 'baujahr', 'lagezurErdoberflaeche',
    'dachart', 'dachgeschossausbau', 'description', 'geometry'
]

In [34]:
def extract_alkis_gdf(gml_dir: str, layer: str = "AX_Gebaeude",
    columns: list[str] | None = None,        # None = keep all; list = keep only these + geometry
    search_token: bytes | None = None,        # None = auto-derive from layer name
    crs: str = "EPSG:25832", print_hits: bool = False, print_progress: bool = True, ) -> gpd.GeoDataFrame:
    """
    Scan all GML/XML files in gml_dir, find those containing `layer`,
    read that layer from each, and return a single concatenated GeoDataFrame.

    Parameters
    ----------
    gml_dir       : directory with ALKIS .xml / .gml files
    layer         : ALKIS layer name to extract (see ALKIS_LAYERS)
    columns       : columns to keep in the output (None = all); geometry always included
    search_token  : bytes to grep for when scanning files; defaults to layer name encoded
    crs           : CRS to assign (data has none stored); ALKIS Hamburg = EPSG:25832
    print_hits    : print filenames that contain the layer
    print_progress: print a progress line per file
    """
    token = search_token or layer.encode()

    # --- 1. find files containing the layer ---
    hits = []
    for fname in sorted(os.listdir(gml_dir)):
        if not fname.endswith((".xml", ".gml")):
            continue
        fpath = os.path.join(gml_dir, fname)
        found = False
        with open(fpath, "rb") as f:
            while not found:
                chunk = f.read(512_000)
                if not chunk:
                    break
                if token in chunk:
                    found = True
        if found:
            hits.append(fpath)

    if print_hits:
        print(f"Files containing '{layer}': {len(hits)}")
        for h in hits:
            print(f"  {os.path.basename(h)}")

    # --- 2. read layer from each file and collect ---
    frames = []
    for i, fpath in enumerate(hits, 1):
        if print_progress:
            print(f"[{i}/{len(hits)}] {os.path.basename(fpath)}", end=" ... ")
        try:
            gdf = gpd.read_file(fpath, layer=layer)

            # column selection (always keep geometry)
            if columns is not None:
                keep = [c for c in columns if c in gdf.columns]
                if "geometry" not in keep:
                    keep.append("geometry")
                gdf = gdf[keep]

            frames.append(gdf)
            if print_progress:
                print(f"{len(gdf)} rows")
        except Exception as e:
            if print_progress:
                print(f"SKIPPED ({e})")

    if not frames:
        raise ValueError(f"No data found for layer '{layer}' in {gml_dir}")

    # --- 3. concatenate and assign CRS ---
    result = pd.concat(frames, ignore_index=True)
    result = gpd.GeoDataFrame(result, geometry="geometry")
    result = result.set_crs(crs)

    print(f"\nDone. Total rows: {len(result):,} | Columns: {list(result.columns)}")
    return result

In [35]:
gdf = extract_alkis_gdf(gml_dir)
print(gdf.shape)

[1/229] HmbTG_ALKIS_260115_031von283.xml ... 769 rows
[2/229] HmbTG_ALKIS_260115_032von283.xml ... 251 rows
[3/229] HmbTG_ALKIS_260115_033von283.xml ... 366 rows
[4/229] HmbTG_ALKIS_260115_035von283.xml ... 23 rows
[5/229] HmbTG_ALKIS_260115_036von283.xml ... 60 rows
[6/229] HmbTG_ALKIS_260115_037von283.xml ... 572 rows
[7/229] HmbTG_ALKIS_260115_039von283.xml ... 707 rows
[8/229] HmbTG_ALKIS_260115_040von283.xml ... 854 rows
[9/229] HmbTG_ALKIS_260115_041von283.xml ... 1061 rows
[10/229] HmbTG_ALKIS_260115_042von283.xml ... 608 rows
[11/229] HmbTG_ALKIS_260115_043von283.xml ... 30 rows
[12/229] HmbTG_ALKIS_260115_044von283.xml ... 10 rows
[13/229] HmbTG_ALKIS_260115_045von283.xml ... 2 rows
[14/229] HmbTG_ALKIS_260115_046von283.xml ... 2 rows
[15/229] HmbTG_ALKIS_260115_047von283.xml ... 764 rows
[16/229] HmbTG_ALKIS_260115_048von283.xml ... 3260 rows
[17/229] HmbTG_ALKIS_260115_049von283.xml ... 4937 rows
[18/229] HmbTG_ALKIS_260115_050von283.xml ... 339 rows
[19/229] HmbTG_ALKIS_260

In [40]:
# find ALL columns that pyarrow can't handle
bad_cols = []
for col in gdf.columns:
    if col == "geometry":
        continue
    try:
        pa.array(gdf[col].tolist(), from_pandas=True)
    except (pa.ArrowInvalid, pa.ArrowTypeError) as e:
        bad_cols.append((col, str(e)[:80]))

print("Problematic columns:")
for col, err in bad_cols:
    print(f"  {col}: {err}")
    print(f"    types: {gdf[col].apply(type).value_counts().to_dict()}")

Problematic columns:
  baujahr: cannot mix list and non-list, non-null values
    types: {<class 'NoneType'>: 219220, <class 'numpy.ndarray'>: 161981, <class 'float'>: 496, <class 'int'>: 7}
  CharacterString: Could not convert 'Öffentl. bestellter Vermessungsing.' with type str: tried to 
    types: {<class 'float'>: 360897, <class 'str'>: 20807}
  name: Expected bytes, got a 'numpy.ndarray' object
    types: {<class 'float'>: 371109, <class 'NoneType'>: 8947, <class 'str'>: 1511, <class 'numpy.ndarray'>: 137}
  weitereGebaeudefunktion: Could not convert array([1060], dtype=int32) with type numpy.ndarray: tried to c
    types: {<class 'float'>: 322020, <class 'NoneType'>: 54813, <class 'numpy.ndarray'>: 4871}
  qualitaetsangaben|AX_DQMitDatenerhebung|herkunft|LI_Lineage|processStep|LI_ProcessStep|processor|CI_ResponsibleParty|individualName|CharacterString: Could not convert '02000' with type str: tried to convert to double
    types: {<class 'float'>: 379749, <class 'str'>: 1955}


In [47]:
def extract_baujahr_scalar(val):
    if val is None:
        return None
    if isinstance(val, np.ndarray):
        return int(val[0]) if len(val) > 0 else None
    if isinstance(val, float):
        return None if np.isnan(val) else int(val)
    if isinstance(val, int):
        return val
    return None

unique_years = (
    gdf["baujahr"]
    .apply(extract_baujahr_scalar)
    .dropna()
    .astype(int)
    .unique()
)
unique_years_sorted = sorted(unique_years)

print(f"Total unique Baujahre: {len(unique_years_sorted)}")
print(unique_years_sorted)


Total unique Baujahre: 229
[np.int64(1), np.int64(1351), np.int64(1398), np.int64(1500), np.int64(1550), np.int64(1561), np.int64(1564), np.int64(1580), np.int64(1600), np.int64(1653), np.int64(1657), np.int64(1660), np.int64(1680), np.int64(1682), np.int64(1683), np.int64(1688), np.int64(1690), np.int64(1692), np.int64(1695), np.int64(1697), np.int64(1700), np.int64(1707), np.int64(1732), np.int64(1736), np.int64(1753), np.int64(1754), np.int64(1757), np.int64(1769), np.int64(1770), np.int64(1780), np.int64(1785), np.int64(1790), np.int64(1800), np.int64(1803), np.int64(1804), np.int64(1808), np.int64(1812), np.int64(1814), np.int64(1815), np.int64(1820), np.int64(1823), np.int64(1825), np.int64(1830), np.int64(1834), np.int64(1835), np.int64(1837), np.int64(1838), np.int64(1840), np.int64(1842), np.int64(1843), np.int64(1845), np.int64(1846), np.int64(1847), np.int64(1848), np.int64(1850), np.int64(1852), np.int64(1854), np.int64(1855), np.int64(1856), np.int64(1857), np.int64(1858),

In [41]:
KEEP_COLUMNS = [
    'identifier', 'beginnt', 'advStandardModell', 'gebaeudefunktion',
    'anzahlDerOberirdischenGeschosse', 'dachform', 'grundflaeche',
    'grundflaeche_uom', 'bauweise', 'baujahr', 'anzahlDerUnterirdischenGeschosse',
    'dachart', 'name', 'weitereGebaeudefunktion', 'art', 'hochhaus', 'geometry'
]


--- identifier ---
identifier
<class 'str'>    381704
{'str': ['urn:adv:oid:DEHHALKAn0000WjZ', 'urn:adv:oid:DEHHALKAn0000hoI']}

--- beginnt ---
beginnt
<class 'str'>    381704
{'str': ['2012-03-28T08:54:41Z', '2022-05-17T12:06:44Z']}

--- advStandardModell ---
advStandardModell
<class 'str'>    381704
{'str': ['DLKM', 'DLKM']}

--- gebaeudefunktion ---
gebaeudefunktion
<class 'int'>    381704
{'int': [2463, 1010]}

--- anzahlDerOberirdischenGeschosse ---
anzahlDerOberirdischenGeschosse
<class 'float'>    381704
{'float': [1.0, 2.0]}

--- dachform ---
dachform
<class 'float'>    381704
{'float': [3100.0, 3100.0]}

--- grundflaeche ---
grundflaeche
<class 'float'>    381704
{'float': [60.0, 75.0]}

--- grundflaeche_uom ---
grundflaeche_uom
<class 'str'>      381702
<class 'float'>         2
{'str': ['urn:adv:uom:m2', 'urn:adv:uom:m2'], 'float': [nan, nan]}

--- bauweise ---
bauweise
<class 'float'>    381704
{'float': [nan, 1100.0]}

--- baujahr ---
baujahr
<class 'NoneType'>         2

AttributeError: 'property' object has no attribute 'get'

In [42]:
def _extract_scalar(val, dtype=None):
    """Reduce None / ndarray / scalar to a single value."""
    if val is None:
        return float("nan")
    if isinstance(val, np.ndarray):
        return val[0] if len(val) > 0 else float("nan")
    return val

def save_gdf_to_parquet(gdf: gpd.GeoDataFrame, path: str) -> None:
    gdf = gdf.copy()

    # drop everything not in KEEP_COLUMNS
    drop_cols = [c for c in gdf.columns if c not in KEEP_COLUMNS]
    if drop_cols:
        print(f"Dropping columns: {drop_cols}")
    gdf = gdf[KEEP_COLUMNS]

    # baujahr: None | ndarray[int32] | float | int → float64
    def clean_baujahr(val):
        if val is None:
            return float("nan")
        if isinstance(val, np.ndarray):
            return float(val[0]) if len(val) > 0 else float("nan")
        return float(val)

    gdf["baujahr"] = gdf["baujahr"].apply(clean_baujahr)

    # name: None | float(nan) | str | ndarray[str] → str | None
    def clean_name(val):
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return None
        if isinstance(val, np.ndarray):
            return str(val[0]) if len(val) > 0 else None
        return str(val)

    gdf["name"] = gdf["name"].apply(clean_name)

    # weitereGebaeudefunktion: None | float(nan) | ndarray[int32] → float64
    def clean_weitere(val):
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return float("nan")
        if isinstance(val, np.ndarray):
            return float(val[0]) if len(val) > 0 else float("nan")
        return float(val)

    gdf["weitereGebaeudefunktion"] = gdf["weitereGebaeudefunktion"].apply(clean_weitere)

    gdf.to_parquet(path)
    print(f"Saved {len(gdf):,} rows → {path}")

In [43]:
gdf_full = gdf.copy()
save_gdf_to_parquet(gdf_full, "buildings_cleaned_up.parquet")

Dropping columns: ['AX_LI_ProcessStep_MitDatenerhebung_Description', 'AX_Datenerhebung', 'CharacterString', 'CI_RoleCode', 'qualitaetsangaben|AX_DQMitDatenerhebung|herkunft|LI_Lineage|processStep|LI_ProcessStep|processor|CI_ResponsibleParty|organisationName|CharacterString', 'qualitaetsangaben|AX_DQMitDatenerhebung|herkunft|LI_Lineage|processStep|LI_ProcessStep|processor|CI_ResponsibleParty|individualName|CharacterString', 'lageZurErdoberflaeche', 'zeigtAufExternes|AA_Fachdatenverbindung|fachdatenobjekt|AA_Fachdatenobjekt|name']
Saved 381,704 rows → buildings_cleaned_up.parquet


In [46]:
gdf_read = gpd.read_parquet("buildings_cleaned_up.parquet")
print(gdf_read.shape)
print(gdf_read.crs)
print(gdf_read.columns.tolist())
print(gdf_read.head())

(381704, 17)
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "ETRS89 / UTM zone 32N", "base_crs": {"name": "ETRS89", "datum_ensemble": {"name": "European Terrestrial Reference System 1989 ensemble", "members": [{"name": "European Terrestrial Reference Frame 1989"}, {"name": "European Terrestrial Reference Frame 1990"}, {"name": "European Terrestrial Reference Frame 1991"}, {"name": "European Terrestrial Reference Frame 1992"}, {"name": "European Terrestrial Reference Frame 1993"}, {"name": "European Terrestrial Reference Frame 1994"}, {"name": "European Terrestrial Reference Frame 1996"}, {"name": "European Terrestrial Reference Frame 1997"}, {"name": "European Terrestrial Reference Frame 2000"}, {"name": "European Terrestrial Reference Frame 2005"}, {"name": "European Terrestrial Reference Frame 2014"}, {"name": "European Terrestrial Reference Frame 2020"}], "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flatten